In [ ]:
from tbh.paths import REPO_ROOT_PATH, DATA_FOLDER
analysis_path = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "51291601_dynamic_mixing" / "task_1"

In [ ]:
import tbh.plotting as pl
import tbh.runner_tools as rt
from tbh.plotting import title_lookup
import pandas as pd
from matplotlib import pyplot as plt 
plt.style.use("ggplot")

In [ ]:
intervention_scenarios = [sc.sc_id for sc in rt.SCENARIOS]
all_scenarios = ['baseline'] + intervention_scenarios
unc_dfs = {
    sc: pd.read_parquet(analysis_path / f"uncertainty_df_{sc}.parquet") for sc in all_scenarios
}

In [ ]:
import yaml

with open(analysis_path / "details.yaml" , "r") as f:
    docs = list(yaml.safe_load_all(f))

model_config = docs[1]

In [ ]:
from tbh.model import get_tb_model
from estival.model import BayesianCompartmentalModel

params, priors, tv_params = rt.get_parameters_and_priors()

model = get_tb_model(model_config, tv_params)
bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)

In [ ]:
# =========================
# COMBINED FIGURE SCRIPT
# =========================

from math import ceil
from copy import copy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

plt.style.use("ggplot")

# -------------------------
# GLOBAL STYLE CONFIG
# -------------------------
PLOT_STYLE = {
    "legend_fs": 12,
    "title_fs": 12,
    "label_fs": 12,
    "tick_fs": 12
}

# -------------------------
# MODEL FIT PLOTTING
# -------------------------
def plot_model_fit_with_uncertainty(
    axis,
    uncertainty_df,
    output_name,
    bcm,
    include_legend=True,
    x_lim=None,
    style=PLOT_STYLE,
    title_lookup=None
):
    df = uncertainty_df[output_name]
    if x_lim:
        df = df.loc[x_lim[0]:x_lim[1]]

    if output_name in bcm.targets:
        t = copy(bcm.targets[output_name].data)
        axis.scatter(list(t.index), t, marker=".", color='black',
                     label='observations', zorder=11, s=30.)

    colour = (0.2, 0.2, 0.8)
    time = df.index

    axis.plot(time, df['0.5'], color=colour, zorder=10, label="model (median)")
    axis.fill_between(time, df['0.25'], df['0.75'],
                      color=colour, alpha=0.5, label="model (IQR)")
    axis.fill_between(time, df['0.025'], df['0.975'],
                      color=colour, alpha=0.3, label="model (95% CI)")

    title = output_name if (title_lookup is None or output_name not in title_lookup) else title_lookup[output_name]

    axis.set_ylabel(title, fontsize=style["label_fs"])
    axis.tick_params(axis='both', labelsize=style["tick_fs"])

    ymin, ymax = axis.get_ylim()
    axis.set_ylim(0., 1.2 * ymax)

    if include_legend:
        axis.legend(fontsize=style["legend_fs"],
            borderpad=0.3,
            loc="lower right"
        )


def plot_all_model_fits(
    uncertainty_df,
    bcm,
    axes,
    excluded_outputs=None,
    style=PLOT_STYLE,
    title_lookup=None
):
    if excluded_outputs is None:
        excluded_outputs = []

    selected_outputs = list(bcm.targets.keys())
    selected_outputs = [o for o in selected_outputs if o not in excluded_outputs]

    for i, output in enumerate(selected_outputs):
        if i >= len(axes):
            break

        ax = axes[i]
        out_name = output if (title_lookup is None or output not in title_lookup) else title_lookup[output]

        x_min = 1990 if output == "notifications" else 2010

        plot_model_fit_with_uncertainty(
            ax,
            uncertainty_df,
            output,
            bcm,
            include_legend=(i == 0),
            x_lim=(x_min, 2025),
            style=style,
            title_lookup=title_lookup
        )

        # ax.set_title(out_name, fontsize=style["title_fs"])

        # Custom x-ticks for first five panels
        if i < 5:

            ax.set_xticks([2010, 2015, 2020, 2025])

    # Hide unused axes
    for j in range(len(selected_outputs), len(axes)):
        axes[j].set_visible(False)


# -------------------------
# AGE-SPECIFIC BOXPLOT
# -------------------------
def plot_age_spec_tbi_prev(
    unc_df,
    bcm,
    ax,
    style=PLOT_STYLE,
    title_lookup=None
):
    agegroups = ["3_9", "10", "15+", "18+"]

    box_data = []
    targets = []
    x_tick_labels = []

    for i_age, age in enumerate(agegroups):
        output_name = f"tst_posXage_{age}_perc"

        year = bcm.targets[output_name].data.index[0]
        quantiles = unc_df[output_name].loc[year]
        target = bcm.targets[output_name].data.iloc[0]

        box_data.append([
            quantiles['0.025'],
            quantiles['0.25'],
            quantiles['0.5'],
            quantiles['0.75'],
            quantiles['0.975']
        ])
        targets.append(target)

        if age == "3_9":
            x_tick_labels.append("3-9" )
        elif age == "15+":
            x_tick_labels.append("15+" )
        elif age == "18+":
            x_tick_labels.append("18+")
        else:
            next_age = agegroups[i_age + 1].replace("+", "")
            x_tick_labels.append(f"{age}-{int(next_age) - 1}")

    bp = ax.bxp(
        [
            {
                'med': d[2],
                'q1': d[1],
                'q3': d[3],
                'whislo': d[0],
                'whishi': d[4],
                'fliers': []
            } for d in box_data
        ],
        positions=range(len(agegroups)),
        showfliers=False,
        patch_artist=True
    )

    for box in bp['boxes']:
        box.set(facecolor='lightblue', alpha=0.6, edgecolor='navy')
    for whisker in bp['whiskers']:
        whisker.set(color='navy', linewidth=1)
    for cap in bp['caps']:
        cap.set(color='navy', linewidth=1)
    for median in bp['medians']:
        median.set(color='darkblue', linewidth=2)

    ax.scatter(range(len(agegroups)), targets,
               color='red', marker='x', s=80, label='Observed')

    model_patch = mpatches.Patch(
        facecolor='lightblue', edgecolor='navy', alpha=0.6,
        label='Modelled\n(quantiles)'
    )
    obs_marker = mlines.Line2D(
        [], [], color='red', marker='x', linestyle='None',
        markersize=8, label='Observed'
    )

    ax.set_xticks(range(len(agegroups)))
    ax.set_xticklabels(x_tick_labels)

    ax.set_xlabel("Age group (years)", fontsize=style["label_fs"])
    ylabel = "TST positivity (%)" if title_lookup is None else title_lookup.get("tst_pos_perc", "TST positivity (%)")
    ax.set_ylabel(ylabel, fontsize=style["label_fs"])

    # ax.set_title(
    #     f"Observed vs modelled TST positivity % by age",
    #     fontsize=style["title_fs"]
    # )

    ax.legend(handles=[model_patch, obs_marker], fontsize=style["legend_fs"], loc="upper left")
    ax.tick_params(axis='both', labelsize=style["tick_fs"])

    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)


# -------------------------
# MAIN COMBINED FIGURE
# -------------------------
def make_combined_figure(
    unc_df,
    bcm,
    title_lookup=None,
    style=PLOT_STYLE
):
    fig = plt.figure(figsize=(14, 10))

    outer = gridspec.GridSpec(
        1, 2,
        width_ratios=[1.6, 1],
        wspace=0.15
    )

    # LEFT PANEL (model fits)
    left_spec = gridspec.GridSpecFromSubplotSpec(
        3, 2,
        subplot_spec=outer[0],
        hspace=0.2,
        wspace=0.3
    )

    left_axes = [fig.add_subplot(left_spec[i]) for i in range(6)]

    excluded_outputs = [
        o for o in bcm.targets
        if (o.startswith("tst_pos") and o != "tst_posXage_18+_perc")
        or (o == "mixing_matrix_distance")
    ]

    plot_all_model_fits(
        unc_df,
        bcm,
        axes=left_axes,
        excluded_outputs=excluded_outputs,
        style=style,
        title_lookup=title_lookup
    )

    # RIGHT PANEL
    right_spec = gridspec.GridSpecFromSubplotSpec(
        2, 1,
        subplot_spec=outer[1],
        height_ratios=[1.0, 1.]
    )

    # Empty top
    ax_empty = fig.add_subplot(right_spec[0])
    ax_empty.axis("off")

    # Boxplot bottom
    ax_box = fig.add_subplot(right_spec[1])
    plot_age_spec_tbi_prev(
        unc_df,
        bcm,
        ax=ax_box,
        style=style,
        title_lookup=title_lookup
    )

    return fig

In [ ]:
fig = make_combined_figure(unc_dfs['baseline'], bcm, title_lookup=title_lookup)
plt.tight_layout()
# plt.show()

plt.savefig("ideas_fig3.png", dpi=300, bbox_inches="tight")
plt.savefig("ideas_fig3.pdf", bbox_inches="tight")